In [ ]:
!pip install datasets jiwer librosa

In [ ]:
!pip install faster_whisper ctranslate2

# teamvillagers/titu-stt-bn-conformer-large-finetuned

In [ ]:
# REVISED CELL 1
import subprocess, sys

# Install compatible versions
packages = [
    "numpy<2.0", 
    "nemo_toolkit[asr]", 
    "soundfile", 
    "huggingface_hub"
]

# Install libraries
subprocess.run([sys.executable, "-m", "pip", "install", "-U"] + packages, check=True)

# Install system dependencies
subprocess.run(["apt-get", "install", "-y", "-q", "libsndfile1", "ffmpeg"], check=True)

print("✅ Installation complete. YOU MUST RESTART THE SESSION NOW.")

In [1]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_Token")

In [5]:
import torch
import pandas as pd
import re
import queue
import threading

from tqdm import tqdm
from datasets import Dataset, Audio
from concurrent.futures import ThreadPoolExecutor
from huggingface_hub import hf_hub_download
from jiwer import wer
import nemo.collections.asr as nemo_asr
from omegaconf import OmegaConf

# ======================
# CONFIG
# ======================
MODEL_REPO_ID = "teamvillagers/titu-stt-bn-conformer-large-finetuned"
HF_TOKEN = secret_value_0

BATCH_SIZE = 8
NUM_GPUS = torch.cuda.device_count()

# ======================
# NORMALIZATION
# ======================
def normalize_bn(text):
    if not isinstance(text, str):
        text = str(text)
    text = re.sub(r'[।,!?;:“”\"\'()\[\]{}—–…]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# Safe extractor (fallback)
def extract_text(x):
    if isinstance(x, str):
        return x
    if hasattr(x, "text"):
        return x.text
    if isinstance(x, list) and len(x) > 0:
        return extract_text(x[0])
    return str(x)

# ======================
# LOAD DATASET
# ======================
parquet_url = "https://huggingface.co/datasets/google/xtreme_s/resolve/refs%2Fconvert%2Fparquet/fleurs.bn_in/test/0000.parquet"

df = pd.read_parquet(parquet_url)
ds = Dataset.from_pandas(df)
ds = ds.cast_column("audio", Audio(sampling_rate=16000))

print(f"Loaded {len(ds)} samples")

# ======================
# LOAD MODEL (CORRECT WAY)
# ======================
print("Downloading NeMo model...")
nemo_path = hf_hub_download(
    repo_id=MODEL_REPO_ID,
    filename="titu_bn_finetuned_lighten.nemo",
    token=HF_TOKEN,
)

models = []

for i in range(NUM_GPUS):
    print(f"Loading model on GPU {i}...")

    m = nemo_asr.models.ASRModel.restore_from(
        nemo_path,
        map_location=f"cuda:{i}"
    )

    m = m.eval()
    models.append(m)

# Apply decoding strategy (IMPORTANT)
decoding_cfg = OmegaConf.create({"strategy": "greedy_batch"})
for m in models:
    m.change_decoding_strategy(decoding_cfg)

print(f"Loaded {len(models)} model replicas")

# ======================
# QUEUE SETUP
# ======================
data_queue = queue.Queue()
for i in range(len(ds)):
    data_queue.put(ds[i])

pbar = tqdm(total=len(ds), desc=f"Evaluating ({NUM_GPUS} GPU)")
lock = threading.Lock()

# ======================
# WORKER
# ======================
def eval_worker(gpu_id):
    model = models[gpu_id]

    local_preds = []
    local_refs = []

    while True:
        batch = []
        try:
            for _ in range(BATCH_SIZE):
                batch.append(data_queue.get_nowait())
        except queue.Empty:
            if not batch:
                break

        audio_arrays = [s["audio"]["array"] for s in batch]

        with torch.no_grad():
            preds = model.transcribe(
                audio=audio_arrays,
                batch_size=len(audio_arrays),
                return_hypotheses=False  # ✅ CRITICAL FIX
            )

        refs = [s["transcription"] for s in batch]

        for p, r in zip(preds, refs):
            p_text = extract_text(p)  # safe guard
            local_preds.append(normalize_bn(p_text))
            local_refs.append(normalize_bn(r))

        with lock:
            pbar.update(len(batch))

    return local_preds, local_refs

# ======================
# MULTI-GPU EXECUTION
# ======================
with ThreadPoolExecutor(max_workers=NUM_GPUS) as executor:
    futures = [executor.submit(eval_worker, i) for i in range(NUM_GPUS)]

all_preds = []
all_refs = []

for f in futures:
    p, r = f.result()
    all_preds.extend(p)
    all_refs.extend(r)

pbar.close()

# ======================
# FINAL WER
# ======================
final_wer = wer(all_refs, all_preds)

print("\n" + "="*40)
print(f"RESULTS FOR: {MODEL_REPO_ID}")
print(f"FLEURS Bengali Test WER: {final_wer * 100:.2f}%")
print(f"Total Samples: {len(all_refs)}")
print("="*40)

Loaded 600 samples
Loading model on GPU 0...


[NeMo W 2026-04-14 17:02:11 modelPT:188] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /kaggle/working/manifests/train_manifest.json
    sample_rate: 16000
    batch_size: 8
    shuffle: true
    num_workers: 2
    pin_memory: true
    use_start_end_token: true
    trim_silence: true
    max_duration: 30.0
    min_duration: 0.5
    is_tarred: false
    tarred_audio_filepaths:
    - - /data2/nemo_asr/nemo_asr_set_3.0/bucket1/audio__OP_0..8191_CL_.tar
    - - /data2/nemo_asr/nemo_asr_set_3.0/bucket2/audio__OP_0..8191_CL_.tar
    - - /data2/nemo_asr/nemo_asr_set_3.0/bucket3/audio__OP_0..8191_CL_.tar
    - - /data2/nemo_asr/nemo_asr_set_3.0/bucket4/audio__OP_0..8191_CL_.tar
    - - /data2/nemo_asr/nemo_asr_set_3.0/bucket5/audio__OP_0..8191_CL_.tar
    - - /data2/nemo_asr/nemo_asr_set_3.0/bucket6/audio__OP_0..8191_CL_.tar
    -

Loading model on GPU 1...


[NeMo W 2026-04-14 17:02:14 modelPT:188] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /kaggle/working/manifests/train_manifest.json
    sample_rate: 16000
    batch_size: 8
    shuffle: true
    num_workers: 2
    pin_memory: true
    use_start_end_token: true
    trim_silence: true
    max_duration: 30.0
    min_duration: 0.5
    is_tarred: false
    tarred_audio_filepaths:
    - - /data2/nemo_asr/nemo_asr_set_3.0/bucket1/audio__OP_0..8191_CL_.tar
    - - /data2/nemo_asr/nemo_asr_set_3.0/bucket2/audio__OP_0..8191_CL_.tar
    - - /data2/nemo_asr/nemo_asr_set_3.0/bucket3/audio__OP_0..8191_CL_.tar
    - - /data2/nemo_asr/nemo_asr_set_3.0/bucket4/audio__OP_0..8191_CL_.tar
    - - /data2/nemo_asr/nemo_asr_set_3.0/bucket5/audio__OP_0..8191_CL_.tar
    - - /data2/nemo_asr/nemo_asr_set_3.0/bucket6/audio__OP_0..8191_CL_.tar
    -

Loaded 2 model replicas



Transcribing:   0%|          | 0/1 [00:00<?, ?it/s]

Transcribing:   0%|          | 0/1 [00:00<?, ?it/s]

Transcribing: 100%|██████████| 1/1 [00:00<00:00,  1.39it/s]

Transcribing:   0%|          | 0/1 [00:00<?, ?it/s]

Transcribing: 100%|██████████| 1/1 [00:00<00:00,  2.67it/s]

Transcribing: 100%|██████████| 1/1 [00:00<00:00,  2.26it/s]2.08it/s]

Transcribing:   0%|          | 0/1 [00:00<?, ?it/s]00:28, 19.81it/s]

Transcribing: 100%|██████████| 1/1 [00:01<00:00,  1.17s/it]

Evaluating (2 GPU):   7%|▋         | 40/600 [00:02<00:26, 20.75it/s]

Transcribing: 100%|██████████| 1/1 [00:00<00:00,  1.67it/s]

Transcribing:   0%|          | 0/1 [00:00<?, ?it/s]00:24, 22.46it/s]

Transcribing: 100%|██████████| 1/1 [00:00<00:00,  1.78it/s]

Evaluating (2 GPU):   9%|▉         | 56/600 [00:02<00:23, 23.35it/s]

Transcribing: 100%|██████████| 1/1 [00:00<00:00,  2.01it/s]

Transcribing:   0%|          | 0/1 [00:00<?, ?it/s]00:20, 26.08it/s]

Transcribing: 100%|██████████| 1/1 [00:00<00:00,  2.08


RESULTS FOR: teamvillagers/titu-stt-bn-conformer-large-finetuned
FLEURS Bengali Test WER: 20.03%
Total Samples: 600


# Risalat/whisper-bengali-ct2

In [ ]:
import pandas as pd
from datasets import Dataset, Audio
import queue
import threading
import re
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from faster_whisper import WhisperModel, BatchedInferencePipeline
from jiwer import wer

# ======================
# CONFIGURATION
# ======================
MODEL_NAME = "Risalat/whisper-bengali-ct2"
COMPUTE_TYPE = "float16"
BATCH_SIZE = 16
BEAM_SIZE = 5

def normalize_bn(text):
    """Basic normalization for fair WER calculation."""
    text = re.sub(r'[।,!?;:“”\"\'()\[\]{}—–…]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# ======================
# DATASET LOADING (Direct Parquet via Pandas)
# ======================
print("Bypassing loader scripts... fetching Parquet directly.")

# FLEURS Bengali Test split direct link from HF Parquet export
# This is a standard URL format for HF Parquet exports
parquet_url = "https://huggingface.co/datasets/google/xtreme_s/resolve/refs%2Fconvert%2Fparquet/fleurs.bn_in/test/0000.parquet"

# Load into Pandas first
df = pd.read_parquet(parquet_url)

# Convert to HF Dataset object
ds = Dataset.from_pandas(df)

# Cast the audio column so it decodes correctly
ds = ds.cast_column("audio", Audio(sampling_rate=16000))

print(f"Successfully loaded {len(ds)} samples.")

# Prepare queue for multi-GPU
data_queue = queue.Queue()
for i in range(len(ds)):
    data_queue.put(ds[i])

pbar = tqdm(total=len(ds), desc="Evaluating (2x T4)")
results_lock = threading.Lock()

def eval_worker(gpu_id):
    model = WhisperModel(
        MODEL_NAME,
        device="cuda",
        device_index=gpu_id, 
        compute_type=COMPUTE_TYPE,
        cpu_threads=2
    )
    pipeline = BatchedInferencePipeline(model=model)
    
    local_preds = []
    local_refs = []
    
    while True:
        try:
            sample = data_queue.get_nowait()
        except queue.Empty:
            break
            
        # Audio handling
        audio_data = sample["audio"]["array"].astype("float32")
        
        segments, _ = pipeline.transcribe(
            audio_data,
            language="bn",
            beam_size=BEAM_SIZE,
            batch_size=BATCH_SIZE,
            vad_filter=True
        )
        
        raw_pred = " ".join(seg.text for seg in segments)
        
        local_preds.append(normalize_bn(raw_pred))
        local_refs.append(normalize_bn(sample["transcription"]))
        
        with results_lock:
            pbar.update(1)
            
    return local_preds, local_refs

# Launch Evaluation
with ThreadPoolExecutor(max_workers=2) as executor:
    futures = [executor.submit(eval_worker, i) for i in range(2)]

all_preds = []
all_refs = []

for f in futures:
    p, r = f.result()
    all_preds.extend(p)
    all_refs.extend(r)

pbar.close()

# ======================
# FINAL SCORE
# ======================
if len(all_refs) > 0:
    final_wer = wer(all_refs, all_preds)
    print("\n" + "="*40)
    print(f"RESULTS FOR: {MODEL_NAME}")
    print(f"FLEURS Bengali Test WER: {final_wer * 100:.2f}%")
    print(f"Total Samples Evaluated: {len(all_refs)}")
    print("="*40)

# facebook/mms-1b-fl102

In [ ]:
import pandas as pd
from datasets import Dataset, Audio
import queue
import threading
import re
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from transformers import Wav2Vec2ForCTC, AutoProcessor
import torch
from jiwer import wer

# ======================
# CONFIGURATION
# ======================
MODEL_NAME = "facebook/mms-1b-fl102"
BATCH_SIZE = 1  # CTC models like MMS typically process one long sample at a time
TARGET_LANG = "ben" # Bengali ISO code for MMS

def normalize_bn(text):
    """Basic normalization for fair WER calculation."""
    text = re.sub(r'[।,!?;:“”\"\'()\[\]{}—–…]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# ======================
# DATASET LOADING
# ======================
print("Fetching Parquet directly for MMS evaluation...")
parquet_url = "https://huggingface.co/datasets/google/xtreme_s/resolve/refs%2Fconvert%2Fparquet/fleurs.bn_in/test/0000.parquet"
df = pd.read_parquet(parquet_url)
ds = Dataset.from_pandas(df)
ds = ds.cast_column("audio", Audio(sampling_rate=16000))

# Prepare queue
data_queue = queue.Queue()
for i in range(len(ds)):
    data_queue.put(ds[i])

pbar = tqdm(total=len(ds), desc="Evaluating MMS (2x T4)")
results_lock = threading.Lock()

def eval_worker(gpu_id):
    # Load model and processor per GPU
    processor = AutoProcessor.from_pretrained(MODEL_NAME)
    model = Wav2Vec2ForCTC.from_pretrained(MODEL_NAME).to(f"cuda:{gpu_id}")
    
    local_preds = []
    local_refs = []
    
    while True:
        try:
            sample = data_queue.get_nowait()
        except queue.Empty:
            break
            
        # Audio handling for MMS
        audio_input = sample["audio"]["array"]
        
        # MMS requires language selection
        processor.tokenizer.set_target_lang(TARGET_LANG)
        inputs = processor(audio_input, sampling_rate=16000, return_tensors="pt").to(f"cuda:{gpu_id}")
        
        with torch.no_grad():
            outputs = model(**inputs).logits
        
        ids = torch.argmax(outputs, dim=-1)
        transcription = processor.batch_decode(ids)[0]
        
        local_preds.append(normalize_bn(transcription))
        local_refs.append(normalize_bn(sample["transcription"]))
        
        with results_lock:
            pbar.update(1)
            
    return local_preds, local_refs

# Launch Evaluation
with ThreadPoolExecutor(max_workers=2) as executor:
    futures = [executor.submit(eval_worker, i) for i in range(2)]

all_preds = []
all_refs = []

for f in futures:
    p, r = f.result()
    all_preds.extend(p)
    all_refs.extend(r)

pbar.close()

# ======================
# FINAL SCORE
# ======================
if len(all_refs) > 0:
    final_wer = wer(all_refs, all_preds)
    print("\n" + "="*40)
    print(f"RESULTS FOR: {MODEL_NAME}")
    print(f"FLEURS Bengali Test WER: {final_wer * 100:.2f}%")
    print(f"Total Samples Evaluated: {len(all_refs)}")
    print("="*40)

In [ ]:
import pandas as pd
from datasets import Dataset, Audio
import queue
import threading
import re
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from transformers import AutoProcessor, SeamlessM4Tv2Model
import torch
from jiwer import wer

# ======================
# CONFIGURATION
# ======================
MODEL_NAME = "facebook/seamless-m4t-v2-large"
TARGET_LANG = "ben"  # Bengali code for SeamlessM4T

def normalize_bn(text):
    """Basic normalization for fair WER calculation."""
    text = re.sub(r'[।,!?;:“”\"\'()\[\]{}—–…]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# ======================
# DATASET LOADING
# ======================
print("Fetching Parquet directly for SeamlessM4T evaluation...")
parquet_url = "https://huggingface.co/datasets/google/xtreme_s/resolve/refs%2Fconvert%2Fparquet/fleurs.bn_in/test/0000.parquet"
df = pd.read_parquet(parquet_url)
ds = Dataset.from_pandas(df)
ds = ds.cast_column("audio", Audio(sampling_rate=16000))

# Prepare queue
data_queue = queue.Queue()
for i in range(len(ds)):
    data_queue.put(ds[i])

pbar = tqdm(total=len(ds), desc="Evaluating SeamlessM4T (2x T4)")
results_lock = threading.Lock()

def eval_worker(gpu_id):
    # Load model and processor per GPU
    processor = AutoProcessor.from_pretrained(MODEL_NAME)
    model = SeamlessM4Tv2Model.from_pretrained(MODEL_NAME).to(f"cuda:{gpu_id}")
    
    local_preds = []
    local_refs = []
    
    while True:
        try:
            sample = data_queue.get_nowait()
        except queue.Empty:
            break
            
        # Audio input (M4T uses 16kHz)
        audio_input = sample["audio"]["array"]
        
        # Preprocess and generate
        # tgt_lang must be specified for transcription/translation
        audio_inputs = processor(audio=audio_input, sampling_rate=16000, return_tensors="pt").to(f"cuda:{gpu_id}")
        
        with torch.no_grad():
            output_tokens = model.generate(**audio_inputs, tgt_lang=TARGET_LANG, generate_speech=False)
        
        # Decode the first (and only) result in the batch
        transcription = processor.decode(output_tokens[0].tolist(), skip_special_tokens=True)
        
        local_preds.append(normalize_bn(transcription))
        local_refs.append(normalize_bn(sample["transcription"]))
        
        with results_lock:
            pbar.update(1)
            
    return local_preds, local_refs

# Launch Evaluation
with ThreadPoolExecutor(max_workers=2) as executor:
    futures = [executor.submit(eval_worker, i) for i in range(2)]

all_preds = []
all_refs = []

for f in futures:
    p, r = f.result()
    all_preds.extend(p)
    all_refs.extend(r)

pbar.close()

# ======================
# FINAL SCORE
# ======================
if len(all_refs) > 0:
    final_wer = wer(all_refs, all_preds)
    print("\n" + "="*40)
    print(f"RESULTS FOR: {MODEL_NAME}")
    print(f"FLEURS Bengali Test WER: {final_wer * 100:.2f}%")
    print(f"Total Samples Evaluated: {len(all_refs)}")
    print("="*40)